In [ ]:
import duckdb

In [ ]:
import chardet

In [ ]:
import io

In [ ]:
import json

In [ ]:
import re

In [ ]:
import os
os.getcwd()

In [ ]:
# 1. Detect encoding
with open("../data/raw/ABS_Waste_Account_Table9_15-11-26.csv", 'rb') as f:
    raw = f.read()

encoding = chardet.detect(raw)['encoding']

print(result)
# {'encoding': 'Windows-1252', 'confidence': 0.73, 'language': '', 'mime_type': 'text/plain'}


In [ ]:
# 2. Pass detected encoding to DuckDB
encoding = result['encoding']
utf8_bytes = raw.decode(encoding).encode('utf-8')

# point DuckDB straight at the file — no import step
df = duckdb.read_csv(io.BytesIO(utf8_bytes))
print(duckdb.sql("DESCRIBE SELECT * FROM df"))
print(duckdb.sql("SELECT * FROM df LIMIT 10"))

In [ ]:
print(duckdb.sql("SELECT Units, COUNT(*) AS n FROM df GROUP BY Units"))

In [ ]:
print(duckdb.sql('SELECT "Table 9. Waste summary, by industry, 2016-17 to 2018-19" as waste_summary, Units FROM df ORDER BY Units'))

In [ ]:
print(duckdb.sql('''CREATE TABLE tidy3 AS \
SELECT "Table 9. Waste summary, by industry, 2016-17 to 2018-19" as waste_summary, units, '2016-17' AS year, "2016-17" AS value \
FROM df \
UNION ALL \
SELECT "Table 9. Waste summary, by industry, 2016-17 to 2018-19" as waste_summary, units, '2017-18' AS year, "2017-18" AS value \
FROM df \
UNION ALL \
SELECT "Table 9. Waste summary, by industry, 2016-17 to 2018-19" as waste_summary, units, '2018-19' AS year, "2018-19" AS value \
FROM df \
'''));

In [ ]:
print(duckdb.sql('SELECT * FROM tidy3'))

In [ ]:
print(duckdb.sql('''SELECT * FROM tidy3 WHERE waste_summary LIKE 'Total waste generated%'
'''))

In [ ]:
rows = duckdb.sql('''SELECT * FROM tidy3 WHERE waste_summary LIKE 'Total waste generated%'
''').fetchall()

In [ ]:
print(rows)

In [ ]:
data = []
for row in rows:
    label, unit, period, value = row
    total_waste_generated, industry = label.split(" - ")
    industry_clean = re.sub(r"\s*\([^)]*\)\s*$", "", industry) # Remove footnotes (a)
    value_clean = round(value) # Remove extra decimal places
    data.append({"industry": industry_clean, "unit": unit, "period": period, "value": value_clean })

print (data)

In [ ]:
result = data
with open("../data/processed/totals.json", "w") as f:
    json.dump(result, f, indent=2)
    